# Full 3-Stage Anti-Spoofing Pipeline Demo

This notebook does NOT train — loads pre-trained checkpoints and runs inference.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","onnxruntime","ultralytics"],check=True)

In [ ]:
import torch, os
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_DIR = "/kaggle/input/antispoof-checkpoints" if os.path.exists("/kaggle") else "./checkpoints"
print(f"Device: {DEVICE}")

In [ ]:
# Load pre-trained checkpoints
import sys; sys.path.insert(0, ".")
from training.config import PipelineConfig, EARConfig

cfg = PipelineConfig(
    ear               = EARConfig(),
    screen_weights    = f"{CKPT_DIR}/best_screen_detector.pt",
    antispoof_weights = f"{CKPT_DIR}/best_antispoof.onnx",
    live_threshold    = 0.52,
    device            = DEVICE,
)

In [ ]:
# Initialize pipeline
from pipeline.antispoof_pipeline import AntiSpoofPipeline
pipeline = AntiSpoofPipeline(cfg)
print("Pipeline ready")

In [ ]:
# Demo on static images with stage-by-stage visualization
import cv2, numpy as np, matplotlib.pyplot as plt

def demo_on_image(img_path):
    frame = cv2.imread(img_path) if img_path else np.zeros((480,640,3),np.uint8)
    result = pipeline.run(frame)
    print(f"Verdict: {result.verdict} | Confidence: {result.confidence:.3f} | Latency: {result.latency_ms:.1f}ms")
    print(f"Stage results: {result.stage_results}")
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    color = {"LIVE":(0,200,0),"SPOOF":(200,0,0),"PENDING":(255,165,0),"NO_FACE":(128,128,128)}[result.verdict]
    plt.imshow(frame_rgb); plt.title(f"{result.verdict} ({result.confidence:.2f})"); plt.axis("off"); plt.show()

demo_on_image(None)  # blank frame demo

In [ ]:
# Benchmark per-stage latency
results = pipeline.benchmark(n_frames=50)
import pandas as pd
pd.DataFrame([results]).T.rename(columns={0:"value"}).round(2)

In [ ]:
# Final summary
print("\n=== TruePresence Anti-Spoofing Pipeline ===")
print(f"Stage 1 (EAR Blink):    {results["stage1_ear_ms"]:.2f} ms")
print(f"Stage 2 (YOLO Screen):  {results["stage2_screen_ms"]:.2f} ms")
print(f"Stage 3 (CNN Classifier):{results["stage3_cnn_ms"]:.2f} ms")
print(f"Total Pipeline:         {results["total_pipeline_ms"]:.2f} ms")
print(f"Estimated FPS:          {1000/max(results["total_pipeline_ms"],1):.1f}")

## Deployment Instructions


